# Market Regime Model Analysis

This notebook documents the research pipeline. It reads the outputs generated by `scripts/run_research.py` so the notebook and dashboard use the same tested artifacts.

## 1. Objective

Classify the next 20-trading-day SPY market regime using information available at the prediction date, then evaluate realized risk across the predicted regimes.

In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px

root = Path('..')
summary = json.loads((root/'results/metrics/research_summary.json').read_text())
pred = pd.read_csv(root/'results/predictions/oos_predictions.csv', index_col=0, parse_dates=True)
regime = pd.read_csv(root/'results/metrics/regime_summary.csv')
importance = pd.read_csv(root/'results/metrics/feature_importance.csv')
summary

{'best_model': 'Logistic Regression',
 'target_horizon_days': 20,
 'bearish_threshold': 0.0012731533732138177,
 'bullish_threshold': 0.025168481106611646,
 'train_period': ['2015-03-16', '2020-12-02'],
 'validation_period': ['2021-01-04', '2022-12-01'],
 'test_period': ['2023-01-03', '2025-12-02'],
 'model_metrics': {'Logistic Regression': {'validation': {'accuracy': 0.37060041407867494,
    'balanced_accuracy': 0.3696467084317551,
    'macro_precision': 0.3716557261533897,
    'macro_recall': 0.3696467084317551,
    'macro_f1': 0.34861111111111115,
    'bearish_precision': 0.4485981308411215,
    'bearish_recall': 0.23076923076923078,
    'neutral_precision': 0.3020833333333333,
    'neutral_recall': 0.27102803738317754,
    'bullish_precision': 0.36428571428571427,
    'bullish_recall': 0.6071428571428571},
   'test': {'accuracy': 0.3360655737704918,
    'balanced_accuracy': 0.3197729224081513,
    'macro_precision': 0.32037946064622974,
    'macro_recall': 0.3197729224081513,
    'm

## 2. Data Quality

The raw data downloader validates required OHLCV fields, sorted dates, duplicates, positive prices, and non-negative volume before the series are aligned.

In [2]:
pred[['SPY_close','regime','predicted_regime']].describe(include='all')

,SPY_close,regime,predicted_regime
count,732.000000,732.000000,732.000000
mean,525.397815,1.169399,1.094262
std,84.362216,0.839349,0.869372
min,379.380005,0.000000,0.000000
25%,445.352493,0.000000,0.000000
50%,529.945007,1.000000,1.000000
75%,593.797485,2.000000,2.000000
max,687.390015,2.000000,2.000000


## 3. Out-of-Sample Regime Results

In [3]:
pred['predicted_regime'].value_counts().sort_index().rename({0:'Bearish',1:'Neutral',2:'Bullish'})

predicted_regime
Bearish    245
Neutral    173
Bullish    314
Name: count, dtype: int64

## 4. Risk by Predicted Regime

In [4]:
regime

,regime,observations,avg_next_day_return,annualized_volatility,max_drawdown,positive_next_day_pct
0,Bearish,245,0.001180,0.163639,-0.098120,0.555102
1,Neutral,173,0.000646,0.133974,-0.091080,0.606936
2,Bullish,314,0.000690,0.159911,-0.114917,0.571885


## 5. Feature Importance

In [5]:
importance.head(12)

,feature,importance_mean,importance_std
0,price_sma20_ratio,0.034011,0.006572
1,sma20_sma50_ratio,0.026429,0.006030
2,atr_14_pct,0.020811,0.010410
3,ret_20d,0.015724,0.011955
4,dxy_return,0.011467,0.010902
5,macd_signal_gap,0.009935,0.008246
6,tnx_change,0.004971,0.007961
7,vol_5d,0.004224,0.005996
8,volume_change,0.003802,0.005354
9,vix_change,-0.002128,0.008238


## 6. Interpretation

Use the generated metrics rather than pre-written performance claims. In particular, check whether bearish predictions coincide with higher realized volatility or larger drawdowns, and whether the ML overlay improves risk-adjusted outcomes versus Buy & Hold. If it does not, report that result honestly.